# **EXPERIMENT NOTEBOOK**

---

In this notebook you can run experiments on the LIBERO environment using the ActionJEPA VLA trained by us or your version of ActionJEPA. Follow ALL the instructions in the notebook!


## **IMPORT SECTION**

In [ ]:
import sys
import os
current_dir = os.getcwd()
libero_path = os.path.join(current_dir, "LIBERO")
if libero_path not in sys.path:
    sys.path.insert(0, libero_path)
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)


from libero.libero import benchmark
from libero.libero.envs.env_wrapper import ControlEnv
from libero.libero.utils import get_libero_path
import numpy as np
import imageio
import torch
from collections import deque
import random
from model.ActionJEPA import ActionJEPA
from tqdm.notebook import tqdm
import cv2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using the device: {device}")

checkpoints_path = './checkpoints'

## **LIBERO ENVIRONMENT INITIALIZATION**

In [ ]:
# @title LIBERO INITIALIZATION PARAMETERS { run: "auto" }

# @markdown Select the RENDER_CAMERA (the point of view used for rendering the simulation) and the TASK_NAME (the task suite of LIBERO to use to pick a random task from it):
RENDER_CAMERA = "agentview" # @param ["agentview", "robot0_eye_in_hand"]
TASK_NAME = "libero_goal" # @param ["libero_10", "libero_90", "libero_spatial", "libero_object", "libero_goal"]

print(f" Camera initialized on: {RENDER_CAMERA}")
print(f" Task suite selected: {TASK_NAME}")

## **MODEL DEFINITION SECTION**

In [ ]:
# Path of the models V-JEPA 2 Encoder, CLIP Encoder and V-JEPA 2 AC Predictor
model_path = "./results/2026_04_20__16_41/best_model.pth" #@param {type:"string"}

In [ ]:
# Path of the models V-JEPA 2 Encoder, CLIP Encoder and V-JEPA 2 AC Predictor
vjepa_path = os.path.join(checkpoints_path,"facebook/vjepa2-vitg-fpc64-256")
vjepa_pred_path = os.path.join(checkpoints_path,"facebook/jepa-wms/vjepa2_ac_droid.pth.tar/vjepa2_ac_droid.pth.tar")
clip_path = os.path.join(checkpoints_path,"openai/clip-vit-large-patch14")

# Path of your model to test
print("Loading model...")
model = torch.load(model_path, map_location=device)
checkpoint = model['model_state_dict']
config = model['config']


model = ActionJEPA(
    vjepa_encoder_path=vjepa_path,
    vjepa_predictor_path=vjepa_pred_path,
    clip_model_path=clip_path,
    num_frames=config['num_frames'],
    use_backbone = True,
    device=device
)


model.load_state_dict(checkpoint, strict=False)
model.to(device)
model.eval()

## **INFERENCE LOOP**

In [ ]:
print("="*50 + "\n")
print(f"[info] RENDER_CAMERA: {RENDER_CAMERA}\n")

benchmark_dict = benchmark.get_benchmark_dict() # dictionary of the type {"<task_suite_name>": <task_suite_class>} (Ex. "libero_spatial": <class 'libero.libero.benchmark.LIBERO_SPATIAL'>)
task_suite_name = TASK_NAME #"libero_spatial" # can also choose libero_spatial, libero_object, etc.
task_suite = benchmark_dict[task_suite_name]() # task suite is a class that represents a set of different tasks, we'll retrieve a specific task with the task_id
n_tasks = task_suite.get_num_tasks() # number of tasks in the selected benchmark
print(f"[info] The benchmark {TASK_NAME} has {n_tasks} tasks.")


# retrieve a random task
task_id = 2 # id of the task to retrieve
task = task_suite.get_task(task_id) # task is the object with all the information about the specific task
task_name = task.name # the name of the task as "KITCHEN_SCENE_1_put_the_black_bowl_at_the_front_on_the_coffee_table"
task_description = task.language # instruction in natural language ("Es. put the black bowl at the front on the coffee table")
print(f"\n[info] retrieving task {task_id} from suite {task_suite_name}\n")
print(f"[info] task description: {task_description}\n")

# retrieve the BDDL (Behavioral Design Definition Language) file for the task, it is a file containing all the informations about objects and initial and goal state
# (Ex. On Bowl1 Table1 means that the bowl 1 is on the table at initial state for example)
# get_libero_path("bddl_files"): find the directory where BDDL files are stored in your pc (Ex. /home/user/Desktop/LIBERO/libero/libero/./bddl_files )
# task.problem_folder: the folder containing the BDDL file for the specific task (Ex. libero_spatial)
# task.bddl_file: the name of the BDDL file for the specific task (Ex. pick_up_the_black_bowl_between_the_plate_and_the_ramekin_and_place_it_on_the_plate.bddl)
task_bddl_file = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
print(f"[info] BDDL file for the task: {task_bddl_file}\n")


# Args for environment initialization
env_args = {
    "bddl_file_name": task_bddl_file, # path of the BDDL file
    "camera_heights": 128,
    "camera_widths": 128,
    "camera_names": ["agentview"],
    "has_renderer": False, # If True, open the MuJoCo screen to render the env
    "has_offscreen_renderer": True, # If True, save images rendered to create a video
    "use_camera_obs": True, # If True, the "obs" will include camera observations (e.g., RGB images) that can be used to create a video.
    "render_camera": RENDER_CAMERA, # the camera name used for rendering and saving video
    "controller": "OSC_POSE"
}

env = ControlEnv(**env_args) # create the env class
env.seed(0) # set a seed for reproducibility
env.reset() # reset the scene and bring to initial state

# init_states is a list of 50 possible different intial states
# each init_state is a vector of state formed by joint positions, objects positions and orientations, velocities ...
init_states = task_suite.get_task_init_states(task_id) # for benchmarking purpose, we fix the a set of initial states
init_state_id = 0 # Among all 50 initial states you can spawn objects in different ways choosing init_state_id (it can be a int number in [0,49])
env.set_init_state(init_states[init_state_id]) # set the init_state chosen

# Invece di env.action_space, usa:
print(f"Controller type: {env.env.robots[0].controller.name}")
print("="*50 + "\n")

window_size = model.num_frames # the number of frames needed to make inference with the loaded model ActionJEPA
interpolation_methods_list = [cv2.INTER_NEAREST, cv2.INTER_LINEAR, cv2.INTER_AREA, cv2.INTER_CUBIC]
interpolation = interpolation_methods_list[config['interpolation_type']]
frame_buffer = deque(maxlen=window_size) # the frame buffer for making inference with ActionJEPA

# Define the text instruction for the VLA
text_input = task_description

print(f"Task: {task_description}")

init_action = [0.] * 7 # a start action needed for the first step, to have the first observation from the environment
obs, _, _, _ = env.step(init_action) # a first step with a zero action to observe the first frame
first_frame = np.flip(obs["agentview_image"], axis=0)
first_frame = cv2.resize(first_frame, (256, 256), interpolation=interpolation)

# Appending the first frame (copying it)
for _ in range(window_size):
    frame_buffer.append(first_frame)

frames = [] # list used to store frames for video saving

# Loop over the environment to apply actions and collect observations
# Obs is an ordered dict that have information about the ROBOT:
# - "agentview_image": the RGB image from the agent's camera => camera_heights x camera_widths x 3 (ex 1024 x 1024 x 3)
# - "robot0_joint_positions": array of vlues of the robot's joint positions
# - "robot0_joint_pos_cos (or sin)": array of cos (or sin) values of the robot's joint positions
# - "robot0_joint_vel": array of values of the robot's joint velocities
# - "robot0_eef_pos": array of values of the robot's end effector position (x,y,z)
# - "robot0_eef_quat": array of values of the robot's end effector orientation in quaternion (x,y,z,w)
# - "robot0_gripper_qpos": the gripper's position (two values for the two fingers distances from the center, 0 means fully closed)
# - "robot0_gripper_qvel": the gripper's velocity (two values for the two fingers velocities)
# And information about the object. Each object called "objectName" has two arrays:
# - "objectName_pos": array of values of the object's position (x,y,z)
# - "objectName_quat": array of values of the object's orientation in quaternion (x,y,z,w)
# - "objectName_to_robot0_eef_pos": array of values of the object's position relative to the robot's end effector (x,y,z)
# - "objectName_to_robot0_eef_quat": array of values of the object's orientation relative to the robot's end effector (x,y,z,w)

with torch.no_grad():
    for step in tqdm(range(200), desc="🤖 Inference execution"):

        current_frame = np.flip(obs["agentview_image"], axis=0) # get the RGB image from the agent's camera
        current_frame = cv2.resize(current_frame, (256, 256), interpolation=interpolation)
        frames.append(current_frame) # needed to save the video
        frame_buffer.append(current_frame) # add the current frame to the frame buffer

        
        vision_input = list(frame_buffer)
        actor_action_seq, refiner_action_seq = model(text_input, vision_input)
        vla_action = refiner_action_seq[0, 0, :].cpu().numpy()

        obs, reward, done, _ = env.step(vla_action)

        if done:
            print(f"Task completed at step: {step}")
            break

    # At the end the env is closed
    env.close()

    # Save video if required to a task.mp4 file
    if len(frames) > 0:
        imageio.mimsave("task.mp4", frames, fps=60)
        print("Video saved: task.mp4")